In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import List, Optional

import pandas as pd

from utils.postgres_util import (
    sanitize_sql_identifier, 
    read_sql_dataframe,
    get_engine_from_env,
)

from utils.file_io import save_data

from utils.paths import get_paths

from utils.pipeline_config_loader import (
    load_pipeline_config,
    build_truth_config_block,
    set_wandb_dir_from_config,
    export_config_snapshot,
)

---

In [ ]:
# Get Path's Object
paths = get_paths()

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

CONFIG_ROOT = paths.configs

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

CONFIG_STAGE = "synthetic"
CONFIG_DATASET = "pump"
CONFIG_RUN_MODE = "train"
CONFIG_PROFILE = "default"

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####  

CONFIG = load_pipeline_config(
    config_root=CONFIG_ROOT,
    stage=CONFIG_STAGE,
    dataset=CONFIG_DATASET,
    mode=CONFIG_RUN_MODE,
    profile=CONFIG_PROFILE,
    project_root=paths.root,
).data

SYN_CFG = CONFIG["synthetic"]
PATHS = CONFIG["resolved_paths"]
FILENAMES = CONFIG["filenames"]

SYNTHETIC_DATA_PATH = Path(PATHS["data_synthetic_dir"])

SCHEMA = str("capstone")
DATASET_ID = str("pump_synthetic_v1")
RUN_ID = str("premelt_run_001")

CHUNK_SIZE = 100_000
ORDER_BY = "observation_timestamp"


SOURCE_TABLE_NAME = "synthetic_observations_timestamped_stage"

OUTPUT_DIRECTORY = SYNTHETIC_DATA_PATH
OUTPUT_BASE_FILE_NAME = "synthetic_timestamped_export"

---

In [ ]:
def export_synthetic_table_to_csv_parts(
    engine,
    *,
    schema: str = "capstone",
    table_name: str = "synthetic_observations_timestamped_stage",
    output_dir: str | Path = "/workspace/artifacts/exports",
    base_file_name: str = "synthetic_timestamped_export",
    chunk_size: int = 100_000,
    order_by: Optional[str] = "observation_timestamp",
) -> List[Path]:
    """
    Read a synthetic Postgres table using the requested projection/machine_status mapping
    and export the result into multiple CSV parts.

    Returns
    -------
    List[Path]
        Paths to the CSV part files that were written.
    """
    safe_schema = sanitize_sql_identifier(schema)
    safe_table = sanitize_sql_identifier(table_name)
    safe_order_by = sanitize_sql_identifier(order_by) if order_by else None

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    sensor_columns = [f"sensor_{i:02d}" for i in range(52)]
    sensor_select_sql = ",\n        ".join(sensor_columns)

    order_clause = f'\n    ORDER BY "{safe_order_by}"' if safe_order_by else ""

    count_sql = f"""
    SELECT COUNT(*) AS row_count
    FROM "{safe_schema}"."{safe_table}"
    """

    total_rows_df = read_sql_dataframe(engine, count_sql)
    total_rows = int(total_rows_df.loc[0, "row_count"])

    if total_rows == 0:
        print(f"No rows found in {safe_schema}.{safe_table}")
        return []

    created_files: List[Path] = []

    for offset in range(0, total_rows, chunk_size):
        sql = f"""
        SELECT
            dataset_id,
            run_id,
            asset_id,
            observation_timestamp AS timestamp,
            {sensor_select_sql},
            CASE LOWER(COALESCE(phase, 'normal'))
                WHEN 'normal' THEN 'NORMAL'
                WHEN 'broken' THEN 'BROKEN'
                WHEN 'abnormal' THEN 'BROKEN'
                WHEN 'failure' THEN 'BROKEN'
                WHEN 'failed' THEN 'BROKEN'
                WHEN 'fault' THEN 'BROKEN'
                WHEN 'recovering' THEN 'RECOVERING'
                WHEN 'recovery' THEN 'RECOVERING'
                ELSE 'NORMAL'
            END AS machine_status
        FROM "{safe_schema}"."{safe_table}"
        {order_clause}
        LIMIT {int(chunk_size)} OFFSET {int(offset)}
        """

        chunk_df = read_sql_dataframe(engine, sql)

        if chunk_df.empty:
            continue

        part_number = (offset // chunk_size) + 1
        part_file_name = f"{base_file_name}_part_{part_number:03d}.csv"

        output_path = save_data(
            chunk_df,
            output_dir,
            part_file_name,
            index=False,
        )

        created_files.append(output_path)
        print(
            f"[export] wrote part {part_number:03d} | "
            f"rows={len(chunk_df):,} | path={output_path}"
        )

    return created_files

---

In [ ]:
engine = get_engine_from_env()

---

In [ ]:
print("Starting Export")

In [ ]:
exported_files = export_synthetic_table_to_csv_parts(
    engine,
    schema=SCHEMA,
    table_name=SOURCE_TABLE_NAME,
    output_dir=OUTPUT_DIRECTORY,
    base_file_name=OUTPUT_BASE_FILE_NAME,
    chunk_size=CHUNK_SIZE,
    order_by=ORDER_BY,
)

exported_files

---

In [ ]:
# Dispose Engine
engine.dispose()

In [ ]:
print("Export Complete")